# Emotion Recognition from Text (NLP)

This project focuses on building a deep learning model that can recognize human emotions from short text sentences. The dataset used is the Emotions Dataset for NLP from Kaggle, which contains 20,000 labeled text samples across six emotion categories:

* 😃 Joy
* 😢 Sadness
* ❤️ Love
* 😡 Anger
* 😨 Fear
* 😲 Surprise

The workflow of the project includes:

**1. Data Loading & Preprocessing**
* Reading .txt files for training, validation, and testing.
* Cleaning and tokenizing text using Keras Tokenizer.
* Padding sequences to ensure uniform input length.
  
**2. Model Building (LSTM-based Neural Network)**
* Embedding layer to convert words into vector representations.
* LSTM layer for capturing sequential context of words.
* Dense layers with dropout for classification into six emotions.

**3. Training & Evaluation**
* Trained using categorical cross-entropy loss and Adam optimizer.
* Evaluated with accuracy and confusion matrix to analyze performance.
  
**4. Prediction Usage Example**
* A custom function is created to input any text sentence and output the predicted emotion.

  **Example**:

    * Input: “I am feeling very happy today!” → Prediction: Joy
    * Input: “I feel really sad and hopeless” → Prediction: Sadness
      
This project demonstrates how Natural Language Processing (NLP) and Recurrent Neural Networks (RNNs) can be applied to understand and classify emotions in text. While the model performs well on common emotions like joy, anger, sadness, it sometimes struggles with underrepresented categories like love and surprise.

**The project can be extended by:**
* Handling class imbalance through oversampling or class weighting.
* Using Transformer-based models (BERT, DistilBERT) for improved accuracy.
* Deploying the trained model as an API for real-world applications like chatbots or sentiment tracking.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

2025-10-18 12:32:09.890384: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760790730.172599      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760790730.251610      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# 1. Load Dataset (.txt files)


In [2]:
def load_txt(filepath):
    texts = []
    labels = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # avoid empty lines
                text, label = line.split(";")
                texts.append(text)
                labels.append(label)
    return pd.DataFrame({"text": texts, "label": labels})

train_df = load_txt("../input/emotions-dataset-for-nlp/train.txt")
val_df   = load_txt("../input/emotions-dataset-for-nlp/val.txt")
test_df  = load_txt("../input/emotions-dataset-for-nlp/test.txt")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (16000, 2)
Validation shape: (2000, 2)
Test shape: (2000, 2)


# 2. Encode Labels

In [3]:
label_encoder = LabelEncoder()
train_df["label_enc"] = label_encoder.fit_transform(train_df["label"])
val_df["label_enc"]   = label_encoder.transform(val_df["label"])
test_df["label_enc"]  = label_encoder.transform(test_df["label"])

print("Classes:", label_encoder.classes_)

Classes: ['anger' 'fear' 'joy' 'love' 'sadness' 'surprise']


# 3. Tokenization & Padding

In [4]:
max_words = 10000   # vocab size
max_len = 100       # max words per sentence

tokenizer = keras.preprocessing.text.Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

X_train = tokenizer.texts_to_sequences(train_df["text"])
X_val   = tokenizer.texts_to_sequences(val_df["text"])
X_test  = tokenizer.texts_to_sequences(test_df["text"])

X_train = keras.preprocessing.sequence.pad_sequences(X_train, maxlen=max_len)
X_val   = keras.preprocessing.sequence.pad_sequences(X_val, maxlen=max_len)
X_test  = keras.preprocessing.sequence.pad_sequences(X_test, maxlen=max_len)

y_train = train_df["label_enc"].values
y_val   = val_df["label_enc"].values
y_test  = test_df["label_enc"].values

# 4. Build Model (Embedding + LSTM)

In [5]:
model = keras.Sequential([
    layers.Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    layers.SpatialDropout1D(0.3),
    layers.LSTM(128, dropout=0.3, recurrent_dropout=0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(label_encoder.classes_), activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

model.build(input_shape=(None, max_len))
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-10-18 12:32:25.888108: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 100, 128)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,420,230 (5.42 MB)

 Trainable params: 1,420,230 (5.42 MB)

 Non-trainable params: 0 (0.00 B)

# 5. Train Model

In [6]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=23,
    batch_size=64,
    verbose=1
)

Epoch 1/23
250/250 ━━━━━━━━━━━━━━━━━━━━ 45s 157ms/step - accuracy: 0.3261 - loss: 1.6064 - val_accuracy: 0.6135 - val_loss: 1.1119
Epoch 2/23
250/250 ━━━━━━━━━━━━━━━━━━━━ 38s 152ms/step - accuracy: 0.7055 - loss: 0.8448 - val_accuracy: 0.8670 - val_loss: 0.4028
Epoch 3/23
250/250 ━━━━━━━━━━━━━━━━━━━━ 38s 151ms/step - accuracy: 0.8937 - loss: 0.3458 - val_accuracy: 0.9175 - val_loss: 0.2489
Epoch 4/23
250/250 ━━━━━━━━━━━━━━━━━━━━ 38s 152ms/step - accuracy: 0.9349 - loss: 0.2092 - val_accuracy: 0.9150 - val_loss: 0.2430
Epoch 5/23
250/250 ━━━━━━━━━━━━━━━━━━━━ 38s 151ms/step - accuracy: 0.9540 - loss: 0.1393 - val_accuracy: 0.9200 - val_loss: 0.2496
Epoch 6/23
250/250 ━━━━━━━━━━━━━━━━━━━━ 37s 150ms/step - accuracy: 0.9617 - loss: 0.1106 - val_accuracy: 0.9220 - val_loss: 0.2437
Epoch 7/23
250/250 ━━━━━━━━━━━━━━━━━━━━ 38s 153ms/step - accuracy: 0.9651 - loss: 0.0991 - val_accuracy: 0.9230 - val_loss: 0.2278
Epoch 8/23
250/250 ━━━━━━━━━━━━━━━━━━━━ 38s 152ms/step - accuracy: 0.9753 - loss: 0

# 6. Evaluate Model

In [7]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc*100:.2f}%")

Test Accuracy: 91.95%


In [8]:
# Classification report
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step

Classification Report:

              precision    recall  f1-score   support

       anger       0.94      0.90      0.92       275
        fear       0.88      0.89      0.88       224
         joy       0.91      0.97      0.94       695
        love       0.85      0.74      0.79       159
     sadness       0.97      0.96      0.97       581
    surprise       0.76      0.73      0.74        66

    accuracy                           0.92      2000
   macro avg       0.88      0.86      0.87      2000
weighted avg       0.92      0.92      0.92      2000



In [14]:
from sklearn.metrics import classification_report
import numpy as np
import joblib
import pickle

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

# Create a dictionary with all components
model_package = {
    'model': model,
    'label_encoder': label_encoder,
    'input_length': 128,  # or whatever your max_length is
    'classes': label_encoder.classes_,
    'model_type': 'emotion_classifier'
}

# Save everything in one .pkl file
with open('complete_emotion_model.pkl', 'wb') as f:
    pickle.dump(model_package, f)

print("✅ Complete model package saved as complete_emotion_model.pkl")

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step

Classification Report:

              precision    recall  f1-score   support

       anger       0.94      0.90      0.92       275
        fear       0.88      0.89      0.88       224
         joy       0.91      0.97      0.94       695
        love       0.85      0.74      0.79       159
     sadness       0.97      0.96      0.97       581
    surprise       0.76      0.73      0.74        66

    accuracy                           0.92      2000
   macro avg       0.88      0.86      0.87      2000
weighted avg       0.92      0.92      0.92      2000

✅ Complete model package saved as complete_emotion_model.pkl


# 6. Usage (Predict Emotion for new text)

In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_emotion(text):
    # Convert text to sequence
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)
    
    # Predict using your trained model
    pred = model.predict(padded)
    
    # Get label
    label = label_encoder.inverse_transform([np.argmax(pred)])
    return label[0], pred[0]

sample_texts = [
    "I am feeling very happy today!",
    "This makes me so angry!",
    "I’m scared of what will happen tomorrow",
    "what to hell are you doing?",
    "I feel really sad and hopeless",
]

for txt in sample_texts:
    emotion, scores = predict_emotion(txt)
    print(f"Text: {txt}\nPredicted Emotion: {emotion}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Text: I am feeling very happy today!
Predicted Emotion: joy

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Text: This makes me so angry!
Predicted Emotion: anger

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Text: I’m scared of what will happen tomorrow
Predicted Emotion: fear

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Text: what to hell are you doing?
Predicted Emotion: fear

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Text: I feel really sad and hopeless
Predicted Emotion: sadness

